In [ ]:
# This is used to count how long the matrix is generated and converted when using software

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

In [ ]:
# --- Please adjust these paths ---
matrix_script = r"D:\DATA\Documents\Xirka Internship\PME\Transformer\transformer\Python Model + Scripts\matrix_multiplier.py"
bin2hex_script = r"D:\DATA\Documents\Xirka Internship\PME\Transformer\transformer\Python Model + Scripts\bin2hex.py"

# Folder where matrix_multiplier.py writes its .mem files (passed as --out_dir below).
# Point this at the same folder your bin2hex commands used, e.g.:
out_dir = Path(r"D:\DATA\Documents\Xirka Internship\PME\Transformer\transformer\exports\Tested Mem Files\#9 APSIPA Data\data_4 (128x128)")
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
matmul_args = [
    sys.executable, "-u", matrix_script,
    "--task", "matmul",
    "--rows_a", "64",
    "--cols_a", "128",
    "--proj_dim", "128",
    "--display", "float",
    "--integers",
    "--min_val", "0",
    "--max_val", "1",
    "--cores_a", "4",
    "--cores_b", "4",
    "--A_total_bits", "8", "--A_frac_bits", "0",
    "--B_total_bits", "8", "--B_frac_bits", "0",
    "--C_total_bits", "8", "--C_frac_bits", "0",
    "--out_dir", str(out_dir),
]

t0 = time.perf_counter()
result = subprocess.run(matmul_args, capture_output=True, text=True)
t1 = time.perf_counter()
matmul_time = t1 - t0

print(result.stdout)
if result.returncode != 0:
    print("--- STDERR ---")
    print(result.stderr)
    raise RuntimeError("matrix_multiplier.py failed, see stderr above")

print(f"Matrix generation time: {matmul_time:.4f} seconds")

In [ ]:
# (filename, --element-bits) pairs, matching your original commands
files_to_convert = [
    (out_dir / "matrix_A_core.mem", 8),
    (out_dir / "matrix_B_core.mem", 8),
    (out_dir / "matrix_A_row.mem", 2),
    (out_dir / "matrix_B_row.mem", 2),
]

bin2hex_times = {}

for filepath, elem_bits in files_to_convert:
    args = [sys.executable, "-u", bin2hex_script, str(filepath), "--element-bits", str(elem_bits)]
    t0 = time.perf_counter()
    result = subprocess.run(args, capture_output=True, text=True)
    t1 = time.perf_counter()
    elapsed = t1 - t0
    bin2hex_times[filepath.name] = elapsed

    print(result.stdout)
    if result.returncode != 0:
        print("--- STDERR ---")
        print(result.stderr)
        raise RuntimeError(f"bin2hex.py failed on {filepath.name}, see stderr above")

    print(f"{filepath.name}: {elapsed:.4f} seconds")

total_bin2hex_time = sum(bin2hex_times.values())
print(f"\nTotal bin2hex conversion time: {total_bin2hex_time:.4f} seconds")

In [ ]:
software_generation_time = matmul_time + total_bin2hex_time

print("Breakdown:")
print(f"  Matrix generation (matmul):     {matmul_time:.4f} s")
print(f"  bin2hex conversion (4 files):   {total_bin2hex_time:.4f} s")
print(f"  ------------------------------------------")
print(f"  Total software time:            {software_generation_time:.4f} s")